In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
import os
import subprocess
import warnings
warnings.simplefilter('ignore')
import sys
import time

def cache_model_background(path, num_workers=16):
    """Start cache_model in background using safetensors, return immediately."""

    script = f'''
import os, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from safetensors.torch import load_file

path = "{path}"
exts = (".bin", ".pt", ".safetensors")
num_workers = {num_workers}

def warmup_file(fpath):
    if fpath.endswith(".safetensors"):
        data = load_file(fpath, device="cpu")
        # return fpath, sum(t.nbytes for t in data.values())

    # fallback for .bin/.pt
    total = 0
    with open(fpath, "rb") as f:
        while chunk := f.read(256 * 1024 * 1024):
            total += len(chunk)
    return fpath, total

if os.path.isdir(path):
    files = [
        os.path.join(root, name)
        for root, _, names in os.walk(path)
        for name in names
        if name.endswith(exts)
    ]
    files.sort()
else:
    files = [path]

print(f"[cache_model] {{len(files)}} file(s), {{num_workers}} worker(s)")
t0 = time.time()
total_bytes = 0

with ThreadPoolExecutor(max_workers=num_workers) as pool:
    futures = {{pool.submit(warmup_file, f): f for f in files}}
    for i, fut in enumerate(as_completed(futures), 1):
        fpath, n = fut.result()
        total_bytes += n
        print(f"[{{i}}/{{len(files)}}] cached {{os.path.basename(fpath)}}")

elapsed = time.time() - t0
gb = total_bytes / 1024**3
print(f"[cache_model] total ≈ {{gb:.2f}} GB in {{elapsed:.2f}}s")
'''

    proc = subprocess.Popen(
        ["python", "-c", script],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True
    )
    print(f"[cache_model] started in background, pid={proc.pid}")
    return proc

# Usage
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-4b-grids15-sft139-ckpt-v1/transformers/step_4000/1")
proc = cache_model_background("/kaggle/input/datasets/nguyennguyen599/qwen-38-27b-rl-32-nvfp4")
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-8/transformers/unsloth-qwen3.8-27b-nvfp4/1")
proc = cache_model_background("/kaggle/input/models/nguyennguyen599/qwen3-8/transformers/z-lab-qwen3.8-27b-dflash2/1")
proc = cache_model_background("/kaggle/input/datasets/nguyennguyen599/ista-daslab-qwen3-8-flash-next-gsq-rco-gguf")

In [ ]:
import subprocess

proc = subprocess.Popen(
    """
    sudo rm -rf /usr/local/cuda-12.8 ;
    sudo dpkg -i /kaggle/input/datasets/nguyennguyen599/cu13-install/cuda-keyring_1.1-1_all.deb &&
    cd /kaggle/input/datasets/nguyennguyen599/cu13-install &&
    sudo dpkg -i ./*.deb
    """,
    shell=True,
)

print("CUDA installation started.")

In [ ]:
!rm -r /kaggle/working/*

In [ ]:
!cp /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/* /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/patches /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/third_party /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/pipeline_runtime_bootstrap /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/vllm_delta_mhar_plugin/vllm_delta_mhar_plugin /kaggle/working/

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import glob
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

%env UV_OFFLINE=1
if os.path.isdir("/kaggle/input/vllm-wheels/wheels/wheels"):
    WHEELS = Path("/kaggle/input/vllm-wheels/wheels/wheels")
else:
    WHEELS = Path("/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels")

WORKING = Path("/kaggle/working")

os.environ["UV_FIND_LINKS"] = str(WHEELS)
os.environ["UV_OFFLINE"] = "1"

WHEELS = Path("/dev/shm/wheels")

src_dir = os.environ["UV_FIND_LINKS"]
dst_dir = "/dev/shm/wheels"
os.makedirs(dst_dir, exist_ok=True)

items_to_copy = os.listdir(src_dir)

def copy_item(item_name):
    src_path = os.path.join(src_dir, item_name)
    dst_path = os.path.join(dst_dir, item_name)
    try:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            return f"Done copy folder: {item_name}"
        else:
            shutil.copy2(src_path, dst_path)
            return f"Done copy file: {item_name}"
    except Exception as e:
        return f"Error copy {item_name}: {e}"

print(f"Start copy {len(items_to_copy)} folder from {src_dir} to {dst_dir}...")

with ThreadPoolExecutor() as executor:
    results = executor.map(copy_item, items_to_copy)

for result in results:
    print(result)
    
print("done all")
os.environ["UV_OFFLINE"] = "1"

os.environ["UV_FIND_LINKS"] = "/dev/shm/wheels"
%env UV_FIND_LINKS=/dev/shm/wheels

In [ ]:
import os
import glob
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

# Supports both exact paths and wildcard patterns.
copy_patterns = [
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/unsloth-zoo",
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/unsloth",
    "/kaggle/input/datasets/nguyennguyen599/unsloth-main/peft",
    str(WHEELS / "liger_kernel*/liger_kernel"),
    str(WHEELS / "quack-kernels*/quack-kernels")
]

sources = []
for pattern in copy_patterns:
    matches = glob.glob(pattern)

    if not matches:
        print(f"WARNING: no match: {pattern}")
    else:
        sources.extend(map(Path, matches))

def copy_item(src: Path):
    dst = WORKING / src.name

    if src.is_dir():
        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True,
            copy_function=shutil.copyfile,  # Faster: does not copy metadata
        )
    else:
        shutil.copyfile(src, dst)

    return src, dst


# A small worker count is usually fastest on Kaggle storage.
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(copy_item, src) for src in sources]

    for future in as_completed(futures):
        try:
            src, dst = future.result()
            print(f"Copied: {src} -> {dst}")
        except Exception as error:
            print(f"Copy failed: {error}")

!uv pip install --no-build-isolation vllm torch torchvision -U
!uv pip install torchaudio torchcodec -U
!uv pip install mamba-ssm==2.3.2.post1 --no-deps
!uv pip install --no-build-isolation /kaggle/working/unsloth-zoo /kaggle/working/unsloth /kaggle/working/peft
!uv pip install flash_attn causal_conv1d bitsandbytes xformers
!uv pip install ${UV_FIND_LINKS}/transformer_engine-*
!uv pip install apex
!uv pip install redis
!uv pip install --no-build-isolation /kaggle/working/liger_kernel
!uv pip install --no-build-isolation \
    flashinfer-python flashinfer-cubin flashinfer-jit-cache
    # flashinfer-python==0.6.15.post1 flashinfer-cubin==0.6.15.post1 flashinfer-jit-cache==0.6.15.post1

!uv pip install --no-build-isolation /kaggle/working/quack-kernels

!uv pip install transformers==5.20.2

!uv pip install --no-build-isolation --no-deps \
    nvidia-cudnn-cu13==9.24.0.43 nvidia-nvshmem-cu13==3.6.5 nvidia-nccl-cu13==2.30.7
    # nvidia-cudnn-cu13==9.20.0.48 nvidia-nvshmem-cu13==3.6.5 nvidia-nccl-cu13==2.30.7
    # nvidia-cudnn-cu13==9.20.0.48 nvidia-nvshmem-cu13==3.6.5 vidia-nccl-cu13==2.29.7

# !uv pip install nvidia-cutlass-dsl==4.8.0.dev0
# !uv pip install nvidia-cutlass-dsl~=4.7.1 nvidia-cutlass-dsl-libs-cu13==4.7.1
# !uv pip install nvidia-cutlass-dsl~=4.8.0 nvidia-cutlass-dsl-libs-cu13==4.8.0
!uv pip install nvidia-cutlass-dsl~=4.8.1 nvidia-cutlass-dsl-libs-cu13==4.8.1

proc.wait()

if proc.returncode != 0:
    raise RuntimeError("CUDA installation failed")

print("CUDA installation finished.")

In [ ]:
!uv pip uninstall tensorflow jax

In [ ]:
!ps aux | grep vllm

In [ ]:
!rm -r /dev/shm/wheels

In [ ]:
!mkdir /dev/shm/draft_model
!cp /kaggle/input/models/nguyennguyen599/qwen3-8/transformers/z-lab-qwen3.8-27b-dflash2/2/* /dev/shm/draft_model

In [ ]:
import os
os.environ.update({
    "FASTSAFETENSORS_ODIRECT": "0",
    "VLLM_HOST_ALLREDUCE": "1",
    "VLLM_HOST_ALLREDUCE": "1",
    "VLLM_HOST_ALLREDUCE_MODE": "60",
    "VLLM_HOST_ALLREDUCE_PREFETCH_BLOCKS": "16",
    "VLLM_HOST_ALLREDUCE_PREFETCH_MAX_MB": "36",
    "VLLM_GDN_DEFERRED_STATE_COMMIT": "stream",
    "VLLM_GDN_DEFERRED_FULL_GRAPH": "1",
    "VLLM_GDN_DEFERRED_LATE": "1",
    "VLLM_FLASHINFER_FA2_SPEC_DECODE": "1",
    "VLLM_EARLY_SPEC_SEQ_LENS": "1",
    "VLLM_BATCH_SHARD_TOPK": "32",
    "VLLM_GDN_SINGLE_STATE_BLOCK": "1",
    "VLLM_MARLIN_PAD_M": "5120x4352:32",
    "VLLM_DRAFT_VOCAB_FILE": "/kaggle/working/fr32k.json",
    "VLLM_KV_CACHE_GROUP_SIZE": "8",
    "VLLM_SM89_FP8_TILES": "128:4096x5120=t32x128x128_w32x64x64_s3_f,65-80:62080x5120=t80x64x128_w80x32x64_s5_f,128:8704x5120=t16x128x64_w16x64x64_s5_f,128:5120x4352=t80x64x128_w80x32x64_s5_f,65-80:3584x5120=t80x64x128_w80x32x64_s3_f,128:5120x1536=t80x64x128_w80x32x64_s5_f,64:4096x5120=t64x64x128_w32x64x64_s3_s,64:5120x1536=t64x64x128_w32x64x64_s3_s,64:3584x5120=t64x64x128_w32x64x64_s3_s,64:5120x4352=t64x64x128_w32x64x64_s3_s,64:8704x5120=t32x64x128_w16x64x64_s6_f,16:4096x5120=t32x128x128_w32x64x64_s3_f,16:5120x4352=t32x128x128_w32x64x64_s3_f,81-96:62080x5120=t96x64x128_w96x32x64_s4_f,81-96:3584x5120=t96x64x128_w96x32x64_s3_f,81-96:5120x1536=t96x64x128_w96x32x64_s2_f,97-128:5120x1536=t64x128x128_w64x64x64_s2_f",
    "VLLM_SWA_FINE_GRAINED_PREFIX_CACHE": "1",
    "VLLM_PREFIX_CACHE_HOLD_REUSE_STATE": "1",    # keep the prompt-end Mamba/SWA states of a running request
    "VLLM_PREFIX_CACHE_HOLD_DECODE_STATE": "1",   # also keep its latest decode state (needs HOLD_REUSE)
    "VLLM_PREFIX_CACHE_FINE_DECODE_STATE": "1",   # save a state at every 16-token boundary while decoding
})

import os

cudnn_root = "/usr/local/lib/python3.13/dist-packages/nvidia/cudnn"
os.environ["CUDNN_HOME"] = cudnn_root
os.environ["CUDNN_PATH"] = cudnn_root
os.environ["LD_LIBRARY_PATH"] = (
    cudnn_root + "/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
)


qwen38_root = f"/kaggle/working/arc_qwen38_x"
# !ARC_TASK_KEY=0934a4d8,36a08778,981571dc,aa4ec2a5 \
!ARC_RUN_STATUS_PATH=/kaggle/working/arc_run_status.json \
python /kaggle/working/qwen38_pipeline.py \
  --qwen38-recipe vision-repl-audit \
  --model-qwen38 /kaggle/input/models/nguyennguyen599/qwen3-8/transformers/unsloth-qwen3.8-27b-nvfp4/2 \
  --qwen38-draft-model /dev/shm/draft_model \
  --output-root {qwen38_root} \
  --final-submission /kaggle/working/submission.json \
  --end-time {global_end_time} \
  --qwen38-max-seconds 0 \
  --finalization-reserve-seconds 360 \
  --tensor-parallel-size 4 \
  --qwen38-port 18083 \
  --temp-root /tmp/arc_vision \
  --context 128000 \
  --concurrency 20 \
  --gpu-memory-utilization 0.94 \
  --max-num-batched-tokens 2048 \
  --draft-tokens 4 \
  --draft-rejection-method block \
  --draft-quantization auto \
  --load-format fastsafetensors \
  --fastsafetensors-max-gpu-shard-gib 8 \
  --vision-max-pixels 2097152 \
  --loop-gzip-action stop \
  --loop-gzip-window-tokens 1000 \
  --loop-gzip-ratio 8 \
  --loop-gzip-check-tokens 1000 \
  --loop-gzip-confirmations 3 \
  --task-token-budget 70000 \
  --tir-repl-budget 70000 \
  --tir-repl-trajectories 1 \
  --queue-order small-area \
  --vllm-arg=--enable-batch-sharded-sampling \
  '--vllm-arg=--compilation-config.cudagraph_capture_sizes=[1,2,4,5,8,10,15,16,20,24,25,30,32,35,40,45,48,50,55,56,60,64,65,70,72,75,80,88,96,104,112,120,128,136,144,152,160]' \
  --min-request-tokens 30000
# 65536
  # --min-request-tokens 60000 \
# --input-mode text-image

In [ ]:
!ps aux | grep vllm

In [ ]:
# !kill 5710

In [ ]:
# assert 1==0

In [ ]:
# !cat /kaggle/working/arc_4b2qwen38/qwen38/server/server.log

In [ ]:
!du -h -ah --max-depth=1 /kaggle/working/ | sort -rh

In [ ]:
!mkdir /kaggle/tmp
!zip -r /kaggle/tmp/log.tar.gz /kaggle/inference_outputs
!zip -r /kaggle/tmp/log_cs.tar.gz /kaggle/working/arc_4b2qwen38
!zip -r /kaggle/tmp/log_cs_1.tar.gz /kaggle/working/arc_qwen38_x

!cp /kaggle/working/submission.json /kaggle/tmp/submission.json
!rm -r /kaggle/working/*

!cp /kaggle/tmp/submission.json /kaggle/working/submission.json
!cp /kaggle/tmp/log.tar.gz /kaggle/working/log.tar.gz
!cp /kaggle/tmp/log_cs.tar.gz /kaggle/working/log_cs.tar.gz
!cp /kaggle/tmp/log_cs_1.tar.gz /kaggle/working/log_cs_1.tar.gz

# !rm -r /kaggle/working/arc_4b2qwen38
# !rm -r /kaggle/working/arc_qwen38_x